# Checkpoint B: a full agent on a track and a car it has never met

This checkpoint closes **Part 2**.

**The brief.**
Race day.
A track you have never driven, `test-circuit-seed2027`, and a car you have never met: five cars of the perturbed family, sealed (`perturb=1000` … `1004`).
You may measure the track with the carla-like car you know, solve its line, and tune on the five training tracks with the training cars.
You may race the race-day track once: the submission check, then five scored races, after every choice is frozen.

Unit 11 left two findings open.
A margin bought on one track did not carry to another (`test-circuit-seed2026`), and the fraction it shipped was the one with the shortest **expected race time** on its training family.
Today both are put to the test, on a track and on cars that are in no family you have raced.

The tracks and the cars, all built by racekit:

- **the training tracks**, five tracks you may race as often as you like, each with its own line and its own measured grip: `test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`, `test-circuit-seed3` and `test-circuit-seed4`;
- **the training cars**, cars 0–23 of the perturbed family (`carla_like(noise=True, perturb=k)`), whose factors you may read with `perturbation_factors(k)`;
- **the race-day track**, `test-circuit-seed2027`: measure it (Question 1), solve its line and plan it (Question 2), and race it only on race day (Question 5);
- **the race-day cars**, the sealed cars `perturb=1000` … `1004`, raced once each, on race day.

## The rules

- **Your submission is your own Unit 11 submission.**
  The setup cell points `SUBMISSION` at `agents/unit-11.py`, the book's Unit 11 reference, as a stand-in: replace that path with your own Unit 11 file before you begin.
- **Three constants change, and nothing else:** `MU_G` (the race-day track's measured grip, Question 1), `LINE_CONTROL` (its line, Question 2) and `MU_FRACTION` (tuned on the family, Question 4).
  Every other constant stays as your Unit 11 file ships it.
- **The seal is an honour rule.** The sealed cars are built in one place only, the race-day cell of Question 5, inside a race.
  Never inspect them by any route: no `perturbation_factors` on them, no attribute of a car built from them, no re-deriving their factors from racekit's published algorithm, and no race with them before race day.
  Tune on the training cars 0–23 only.
- **No race on `test-circuit-seed2027`** except the six grip probes (Question 1) and race day (Question 5).
  Loading it, solving its line and planning on it are allowed.
  Race day starts with `check_submission` on your frozen file, which races racekit's default kinematic car on the race-day track with the timing check; it must pass before any sealed car races there.
- **Timing.** Every race in this notebook uses a **timing-off copy** of a file, `MPC_BUDGET_MS = None` written in with `write_variant` (the setup's `NO_TIMING`, as in Unit 11), so the numbers reproduce on any machine.
  The frozen file itself keeps `MPC_BUDGET_MS = 5.0`, and it is the file `check_submission` validates.
- **The clean rule** (Unit 9's, from the race's own line): a race is clean when it finished, with 0 collisions, and its max |cross-track error from the line it raced| over the driving rows is at most 2 m.
- **Every three-lap race** uses `carla_like(noise=True, perturb=k)` and `max_seconds=700.0`; the race seed is 0 for every tuning race of Question 4 and k mod 5 for sealed car k.

**The bar:** all five race-day races clean, and their median race time at most **300 s**.
The race is pass/fail on the bar.
Your evidence and your written answers carry the rest of the grade: a submission that misses the bar after an honest Question 4 can still earn full marks, if the write-up says what the record showed.

Run the setup cell first.
Each question names the variables your answer must define; fill in every `None`, and write each written answer in the markdown cell under its prompt.
Run the questions in order: later questions reuse earlier functions and results.
A race takes a few seconds (a perturbed MPC race about 6 s); no cell should run for more than about two minutes, so the long sweeps are split one fraction per cell.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like, perturbation_factors
from racekit.check_submission import check_submission
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
SUBMISSION = AGENTS / "unit-11.py"  # a stand-in: replace it with the path of YOUR Unit 11 submission
PROBE_AGENT = AGENTS / "unit-07.py"  # the book's Unit 7 reference: Unit 8's grip-probe driver (Question 1 only)
workdir = Path(mkdtemp(prefix="roar-checkpoint-b-"))

TRAINING = ("test-circuit", "test-circuit-seed1", "test-circuit-seed2", "test-circuit-seed3", "test-circuit-seed4")
TRAINING_CARS = tuple(range(24))  # the perturbed family's training cars: the only cars you tune on
RACE_DAY_TRACK = "test-circuit-seed2027"  # raced only by the six probes (Question 1) and on race day (Question 5)
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside a race (Unit 11)
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
MAX_SECONDS = 700.0  # s: every three-lap race's time cap
TARGET_S = 300.0  # s: race day's target median
PROTOCOL_SEED = 0  # Question 4: the race seed of every tuning race
START_FRACTION = 0.54  # Unit 11's shipped MU_FRACTION: Question 4 starts here
FRACTION_STEP = 0.02  # the fraction grid (Units 9-11)
CEILING, FLOOR = 0.60, 0.40  # Question 4 never races a fraction above CEILING or below FLOOR
A_DEC, A_ACC, TOP_SPEED = 5.0, 4.0, 45.0  # m/s^2, m/s^2, m/s: the reference's profile (use yours if they differ)
MARGIN, PIN = 2.5, 30  # m kept from each edge, and waypoints at each end of the lap held on the centreline (Unit 9)

# Unit 8's grip probe (Question 1): PROBE_AGENT with A_LAT = PROBE_A_LAT and GRIP 1.0 except PROBE_KNOB in the
# probed section; one lap (laps=1) on the unperturbed carla_like(noise=True), race seed PROBE_SEED, PROBE_SECONDS cap.
PROBE_A_LAT, PROBE_KNOB, PROBE_SEED, PROBE_SECONDS = 3.5, 3.5, 0, 300.0
MIN_ROWS = 20  # a section with this many qualifying rows or fewer gives no estimate

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per training track.
MEASURED_MU_G = {
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2": (9.06, 8.37, 8.49, 8.86, 9.78, 9.92),
    "test-circuit-seed3": (9.55, 8.89, 8.15, 9.94, 8.50, 9.09),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
}

race_log = []  # every race you run with race(), as (track, car): your evidence (car None: the unperturbed car)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track, k=None, seed=0, laps=3, max_seconds=MAX_SECONDS):
    # Every race of this checkpoint: the carla-like car with its noise, car k of the perturbed family
    # (k None: the unperturbed car), one race seed.
    race_log.append((track, k))
    return run_race(source, track, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps,
                    max_seconds=max_seconds)


def submission_module(source):
    # A submission's module as it ships (Unit 9's idiom): its constants are attributes, e.g. .LINE_CONTROL.
    with shim_installed():
        _, module_name = load_submission(source)
    return sys.modules[module_name]


# Units 2 and 9's tools, as Units 9-11 gave them.
def menger(x, y):
    # The signed Menger curvature at every point of a closed polyline (NumPy arrays; left turns positive).
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def spline_basis(n, step=10):
    # The n x (n // step) cubic B-spline basis: a line's offsets are spline_basis(n) @ its control values.
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    # The left unit normal at every point of a closed polyline, as two NumPy arrays (nx, ny).
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    # v[i+1] - 2 v[i] + v[i-1] along the first axis, the ends wrapping round.
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def line_offsets(control, n):
    # A line's offset (m, left positive) at each of the n waypoints, from its control values.
    return spline_basis(n) @ np.array(control, dtype=float)


# Unit 9's clean rule, from the race's own line (offsets: that line's per-waypoint offsets).
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


# Your submission's test-circuit line and grip, as it ships them (test-circuit is raced as shipped).
shipped = submission_module(SUBMISSION)
SHIPPED_LINE = list(shipped.LINE_CONTROL)
SHIPPED_MU_G = list(shipped.MU_G)

## Question 1 - Measure the track

The race-day track's grip is not the grip of any track you have raced, so measure it the way Unit 8 did.

First write Unit 8's probe rule as a function.
`section_p99(telemetry, track, min_rows=MIN_ROWS)` returns a list of six pairs `(rows, p99)`, one per grip section, where:

- |a| at row i (for i from 1 to N − 2) is the size of the second difference of (x, y) at rows i − 1, i, i + 1, divided by 0.05²;
- row i qualifies when rows i − 1, i and i + 1 all have `held == 0`, both steps between them are shorter than 5 m, and `v` at row i is above 5 m/s;
- row i's section is `grip_sections(track)[(progress[i] + 1) % n]` (racekit's progress p is waypoint p + 1);
- `rows` is the number of qualifying rows in the section, and `p99` the 99th percentile of their |a| (`np.percentile(values, 99)`), or `None` when `rows <= min_rows`.

Then run Unit 8's six grip probes on `RACE_DAY_TRACK`, one per section.
The probe for section s is a variant of `PROBE_AGENT` with `A_LAT = PROBE_A_LAT` and `GRIP` 1.0 in every section except `PROBE_KNOB` in section s, raced with `race(source, RACE_DAY_TRACK, seed=PROBE_SEED, laps=1, max_seconds=PROBE_SECONDS)` (the unperturbed car).
From each probe keep only the probed section's pair.

Unit 8's rules for reading a probe:

- a probe that crashed (at least one collision) slid in its section, so its p99 is an **estimate** of μg there;
- a probe that never crashed gives a **lower bound**: the car did not need more grip than that, so the grip is at least that much;
- a section with `MIN_ROWS` qualifying rows or fewer gives **no estimate** (that would need another probe: ask your teacher before you go on).

The planner uses every value as measured, a lower bound included, as Units 9–11 did with `monza-like`'s section 0.

Define:

- `section_p99`: the function;
- `race_day_track`: `load_track(RACE_DAY_TRACK)`;
- `probes`: a dict from each section 0 … 5 to its probe's `RaceResult`;
- `probe_outcome`: a dict from each section to `(finished, collisions)` of its probe;
- `probe_rows`: a dict from each section to the number of qualifying rows in the probed section;
- `race_day_mu_g`: a tuple of the six probed sections' p99, in section order, each rounded to 0.01 (m/s²): the race-day track's `MU_G`;
- `kind`: a dict from each section to `"estimate"`, `"lower bound"` or `"no estimate"`, by Unit 8's rules;
- `probe_races`: the number of entries in `race_log` on `RACE_DAY_TRACK` (it should be exactly 6).

In [ ]:
def section_p99(telemetry, track, min_rows=MIN_ROWS):
    return []  # TODO: six (rows, p99 or None) pairs


race_day_track = None  # TODO
probes = {}  # TODO: the six probes, the only races on RACE_DAY_TRACK before race day
probe_outcome = {}  # TODO
probe_rows = {}  # TODO
race_day_mu_g = None  # TODO
kind = {}  # TODO
probe_races = None  # TODO

**Written answer.**
Which section's value is a lower bound, and why does the probe's outcome make it one?
The planner will use it as if it were the grip: is that a risk for race day, or a cost, and why?
Compare the race-day track's grip with the training tracks' (`MEASURED_MU_G`): where is it lowest?

*Your answer here.*

## Question 2 - Solve its line and plan the lap

Write Unit 9's line solver, `solve_line(track, margin=MARGIN, pin=PIN)`, returning the control values as a NumPy array:

1. the basis `spline_basis(n)` (n × m, m = n // 10), the left normals `normals(track.x, track.y)`, and the smoothness system: A is the rows of `second_difference(nx[:, None] * basis)` stacked **above** the rows of `second_difference(ny[:, None] * basis)`, and b = −(the n entries of `second_difference(track.x)` followed by the n entries of `second_difference(track.y)`);
2. the bounds: control value k's support is the waypoints where column k of the basis is > 0; its bound h_k is the smallest `track.half_width[i] - margin` over its support, except that h_k = 0 when its support contains any of the `pin` waypoints at each end of the lap (indices 0 … pin − 1 and n − pin … n − 1);
3. `Q, R = np.linalg.qr(A)`, then `lsq_linear(R, Q.T @ b, bounds=(-h - 1e-12, h + 1e-12), method="bvls")`, and return its `.x`.

Then write Unit 7's planner for a line, `plan_lap(track, offsets, mu_g, fraction)`, returning the planned time of one lap (s), the way the agents plan:

1. the line's points: x'_i = x_i + `offsets[i]`·nx_i and y'_i = y_i + `offsets[i]`·ny_i, with (nx, ny) the track's left normals;
2. on the line's points: κ_i = `menger(x', y')`, Δs_i the length of the segment from point i to point i + 1 (the last segment closes the lap), and the line's grip sections: point i is in section min(⌊6·d_i / L⌋, 5), with d_i the distance along the line from point 0 to point i and L the line's lap length;
3. the cap of every point, min(`TOP_SPEED`, √(`fraction`·`mu_g[section]` / |κ_i|)) where |κ_i| > 1e-9, and `TOP_SPEED` elsewhere;
4. the backward pass, twice round the lap: for i from n − 1 **down** to 0, v_i = min(v_i, √(v_{(i+1) mod n}² + 2·`A_DEC`·Δs_i));
5. then the forward pass, twice round the lap: for i from 0 **up** to n − 1, v_{(i+1) mod n} = min(v_{(i+1) mod n}, √(v_i² + 2·`A_ACC`·Δs_i));
6. the planned lap time, the sum over the n segments of Δs_i / ((v_i + v_{(i+1) mod n}) / 2).

Plan both tracks at Unit 11's shipped fraction, `START_FRACTION` (0.54): `test-circuit` with your submission's shipped line and grip (`SHIPPED_LINE`, `SHIPPED_MU_G`), and the race-day track with your line and your `race_day_mu_g`.
(Question 5 plans the race-day track again, at the fraction you freeze.)

Define:

- `solve_line`, `plan_lap`: the functions;
- `race_day_control`: your race-day line's control values, as a list of floats (it becomes the frozen file's `LINE_CONTROL`);
- `race_day_offsets`: `line_offsets(race_day_control, len(race_day_track))`;
- `inside_margin`: `True` when every |offset| is at most `race_day_track.half_width - MARGIN` (to within 1e-9);
- `planned_lap_054`: a dict from `"test-circuit"` and `RACE_DAY_TRACK` to the planned lap time at 0.54 (s);
- `fig`, `ax`: the race-day line's offsets against `race_day_track.s` (the distance along the centreline), with ± (half-width − `MARGIN`) drawn for comparison, axis labels and a title.

In [ ]:
def solve_line(track, margin=MARGIN, pin=PIN):
    return None  # TODO: the control values


def plan_lap(track, offsets, mu_g, fraction):
    return None  # TODO: the planned time of one lap (s)


race_day_control = None  # TODO
race_day_offsets = None  # TODO
inside_margin = None  # TODO
planned_lap_054 = {}  # TODO
fig, ax = None, None  # TODO

**Written answer.**
Is the race-day track faster or slower on paper than `test-circuit` at the same fraction, by how much, and why (its length, its corners, its grip)?
A plan says how fast the car is asked to go, not whether it can: in one or two sentences, what can the plan not tell you about racing 0.54 on this track with a car you have never met?

*Your answer here.*

## Question 3 - Two rules, one record

Unit 11 gave you two ways to choose a fraction from a family's record:

- **the expected race time** (Unit 11 §2): the mean race time over all the family's races, each race counted at its time (a race not finished at the cap), and pick the fraction with the shortest;
- **the clean rule and the margin rule** (Units 7 and 11): the **edge** is the highest fraction clean on every race of the family, and the margin rule ships one grid step (`FRACTION_STEP`) inside it.

The cell below is the **reference agent's** record on the track × car family: the probe record of plan `roar-011`, `agents/unit-11.py` on the 24 training cars × the five training tracks (120 races per fraction), each track with its own line and measured grip, race seed k mod 5 for car k, the 700 s cap.
Per fraction it gives the number of clean races and their mean time, and every race that was not clean, as (track, car, race time).
Every not-clean race crashed once, fell back, and finished, under the cap.

Before racing anything, apply both rules to this record.

Write:

- `expected_time(n_clean, clean_mean, not_clean_times)`: the mean over all the races of a fraction, the clean ones at `clean_mean` (their mean is all you need) and each not-clean one at its time.

Define:

- `expected_time`: the function;
- `expected`: a dict from each fraction to its expected race time (s);
- `expected_pick`: the fraction with the shortest expected race time;
- `not_clean_rate`: a dict from each fraction to the share of its races that were not clean;
- `family_edge`: the highest fraction that is clean on all 120 races;
- `margin_pick`: the margin rule's fraction, `round(family_edge - FRACTION_STEP, 2)`.

In [ ]:
FAMILY_RACES = 120  # 24 training cars x 5 training tracks, per fraction
# The probe record of plan roar-011: agents/unit-11.py on the track x car family (race seed k mod 5, 700 s cap).
CLEAN_RECORD = {  # fraction: (clean races, their mean race time in s)
    0.56: (110, 274.50),
    0.54: (118, 276.94),
    0.52: (118, 279.52),
    0.50: (120, 282.21),
    0.48: (120, 285.11),
}
NOT_CLEAN_RECORD = {  # fraction: every race that was not clean, as (track, car, race time in s); all finished
    0.56: [("test-circuit", 8, 484.45), ("test-circuit", 9, 391.90),
           ("test-circuit-seed2", 1, 498.60), ("test-circuit-seed2", 8, 407.25), ("test-circuit-seed2", 9, 319.00),
           ("test-circuit-seed2", 10, 496.30), ("test-circuit-seed2", 13, 499.30),
           ("test-circuit-seed2", 16, 316.95), ("test-circuit-seed2", 17, 500.10),
           ("test-circuit-seed2", 20, 324.75)],
    0.54: [("test-circuit-seed2", 13, 501.00), ("test-circuit-seed2", 16, 413.40)],
    0.52: [("test-circuit-seed2", 9, 508.35), ("test-circuit-seed2", 16, 416.60)],
    0.50: [],
    0.48: [],
}

In [ ]:
def expected_time(n_clean, clean_mean, not_clean_times):
    return None  # TODO: s


expected = {}  # TODO
expected_pick = None  # TODO
not_clean_rate = {}  # TODO
family_edge = None  # TODO
margin_pick = None  # TODO

**Written answer.**
The two rules pick different fractions from the same record.
Given the bar (all five race-day races clean, median at most 300 s), which rule fits race day, and why?
In two or three sentences: what does the expected-time rule assume about how often a car crashes on race day, and what do you know about the race-day track and cars that this record does not show?
Answer before you race anything: this is your choice, and Question 6 asks you to look back at it.

*Your answer here.*

## Question 4 - Tune your agent on the family

The record of Question 3 is the reference's; your agent's edge may be somewhere else.
Find it with the **small protocol** (Unit 11's Exercise 8, on all five training tracks), on **your** submission:

- **the cars:** the three lowest-grip training cars, from `perturbation_factors(k)["grip"]` over `TRAINING_CARS`, lowest first;
- **the order:** the five training tracks in the order of `TRAINING`, and on each track the three cars in that order: 15 races per fraction;
- **each race:** a timing-off copy of your submission at the fraction, raced with `race(source, track, k, seed=PROTOCOL_SEED)`.
  On `test-circuit` only `MU_FRACTION` changes (the shipped line and grip); on every other training track write in that track's line, `solve_line` of it, and its grip, `MEASURED_MU_G[track]`.
  A race is judged by the clean rule against the line it raced;
- **stop each fraction at its first race that is not clean:** the fraction is then not clean on the family, and the races after it would only cost time.

Start at `START_FRACTION` (0.54). **Every outcome has a defined path:**

- **0.54 is not clean:** go down by `FRACTION_STEP` until a fraction is clean on all 15 races: that is the **edge**.
  Ship one step below it.
- **0.54 is clean on all 15:** go up by `FRACTION_STEP` until a fraction is not clean; the edge is the highest clean one; ship one step below it.
  If 0.60 (`CEILING`) is clean on all 15, stop there: the edge is 0.60, and you ship 0.58.
- **Nothing is clean down to 0.40 (`FLOOR`):** ship 0.40 and say so in your written answer.
  Race day is still raced, and the write-up carries the grade.

Race **one fraction per cell**, below.
Each fraction has two cells: the first nine races of `protocol_order` (`test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`), then, only if those nine were clean, the last six (`test-circuit-seed3`, `test-circuit-seed4`).
Three fractions are laid out; if your path needs more, add two cells per fraction in the same pattern.

The book's stand-in, `agents/unit-11.py`, finds the edge at 0.50 and ships 0.48; your agent may not, and that is not an error.

Define, in the first cell:

- `protocol_cars`: the three cars, lowest grip first;
- `protocol_order`: the list of the 15 `(track, car)` pairs in protocol order;
- `training_lines`: a dict from each training track to the line it is raced with (`SHIPPED_LINE` on `test-circuit`, `solve_line` of the track elsewhere, each a list of floats);
- `protocol_source(fraction, track)`: a function returning the path of the timing-off copy of your submission for that fraction and track (`variant(..., MU_FRACTION=fraction, **NO_TIMING)`, plus `LINE_CONTROL` and `MU_G` off `test-circuit`);
- `protocol_races(fraction, pairs)`: a function that races the `(track, car)` pairs in order and stops at the first race that is not clean; it records every race it runs in `protocol_record` and returns `(all_clean, results)`, with `results` the list of the races it ran;
- `protocol_record`: a dict from `(fraction, track, car)` (the fraction rounded to 2 decimals) to `(clean, race time in s, collisions, finished)`, for every race of this question.

In [ ]:
protocol_cars = None  # TODO
protocol_order = []  # TODO
training_lines = {}  # TODO


def protocol_source(fraction, track):
    return None  # TODO: the path of the timing-off copy


def protocol_races(fraction, pairs):
    return None, []  # TODO: (all_clean, results)


protocol_record = {}  # every race of this question, filled by protocol_races

In [ ]:
# TODO: your first fraction, START_FRACTION (0.54): the first nine races of protocol_order only

In [ ]:
# TODO: your first fraction: the last six races of protocol_order, only if the first nine were clean

In [ ]:
# TODO: your second fraction (one step down or up, by your path): the first nine races only

In [ ]:
# TODO: your second fraction: the last six races, only if the first nine were clean

In [ ]:
# TODO: your third fraction: the first nine races only

In [ ]:
# TODO: your third fraction: the last six races, only if the first nine were clean

Now freeze your choice.
Your frozen file is your submission with the three race-day constants written in, and nothing else: `MU_FRACTION`, `LINE_CONTROL` (your `race_day_control`) and `MU_G` (your `race_day_mu_g`); it keeps `MPC_BUDGET_MS` as your file ships it.
Every race-day race uses its timing-off copy.

Define:

- `fraction_clean`: a dict from each fraction you raced to `True` when it was clean on all 15 races;
- `path`: `"down"`, `"up"` or `"floor"`, the path you followed;
- `edge`: the edge (`None` on the `"floor"` path);
- `FROZEN_FRACTION`: the fraction you ship, rounded to 2 decimals;
- `FROZEN_SUBMISSION`: `variant(SUBMISSION, "frozen", MU_FRACTION=FROZEN_FRACTION, LINE_CONTROL=race_day_control, MU_G=race_day_mu_g)`, the file you submit;
- `FROZEN_RACE_COPY`: `variant(FROZEN_SUBMISSION, "frozen-timing-off", **NO_TIMING)`, the file race day races;
- `protocol_races_run`: the number of entries in `protocol_record`.

After this cell your choice is frozen: nothing about it may change once you have raced the race-day track.

In [ ]:
fraction_clean = {}  # TODO
path = None  # TODO
edge = None  # TODO
FROZEN_FRACTION = None  # TODO
FROZEN_SUBMISSION = None  # TODO
FROZEN_RACE_COPY = None  # TODO
protocol_races_run = None  # TODO

**Written answer.**
Report your path, your edge and the fraction you froze, with the race that decided the edge (track, car, what happened).
Compare them with Question 3: does your small protocol find the same edge as the reference's 120-race record, and the same pick as the margin rule there?
In two or three sentences: 15 races on the three lowest-grip cars, one seed each: what could this protocol miss, and what does the margin protect you against that the family cannot show you?

*Your answer here.*

## Question 5 - Race day

Your choice is frozen.
**First plan, then check, then race.**

Plan the race-day track at your frozen fraction with Question 2's `plan_lap`.
Then run `check_submission` on the frozen file, on the race-day track: it races racekit's default kinematic car (not a carla-like car), on the file as shipped, with the timing check (`step()`'s p99 within its budget).
Run it when nothing else is running on your machine.
It must pass before any sealed car races; if it does not, read its report, fix what it names (only the three constants may differ from your Unit 11 submission), freeze again, and check again.

Define:

- `planned_lap_frozen`: `plan_lap(race_day_track, race_day_offsets, race_day_mu_g, FROZEN_FRACTION)` (s);
- `planned_race_frozen`: the planned time of a three-lap race, `3 * planned_lap_frozen` (s);
- `submission_report`: `check_submission(FROZEN_SUBMISSION, track=RACE_DAY_TRACK)`, with `submission_report.ok` `True`.

In [ ]:
planned_lap_frozen = None  # TODO
planned_race_frozen = None  # TODO
submission_report = None  # TODO: check the frozen file before any sealed car races

Now race the five sealed cars, **once**.
This is the only cell that builds a sealed car: race each car k of `SEALED_CARS` with `race(FROZEN_RACE_COPY, RACE_DAY_TRACK, k, seed=k % 5)`, in order, and do nothing else with them.
Race all five even if one is not clean: race day is the whole result.
Run this cell once.

Define:

- `race_day`: the list of the five races, in the order of `SEALED_CARS`;
- `race_day_times`: the list of their race times (s);
- `race_day_errors`: the list of their max line errors from your race-day line, `line_error(result, race_day_offsets)` (m);
- `race_day_clean`: a list of five booleans, `clean(result, race_day_offsets)` for each race;
- `race_day_median`, `race_day_mean`: the median and the mean of the five race times (s);
- `meets_bar`: `True` when all five races are clean and `race_day_median <= TARGET_S`;
- `race_minus_plan`: `race_day_median - planned_race_frozen` (s);
- `race_day_track_races`: the number of entries in `race_log` on `RACE_DAY_TRACK` (it should be exactly 11: the six probes and the five sealed races).

In [ ]:
SEALED_CARS = range(1000, 1005)
race_day = []  # TODO: race FROZEN_RACE_COPY on RACE_DAY_TRACK, once per sealed car, race seed k % 5
race_day_times = []  # TODO
race_day_errors = []  # TODO
race_day_clean = []  # TODO
race_day_median, race_day_mean = None, None  # TODO
meets_bar = None  # TODO
race_minus_plan = None  # TODO
race_day_track_races = None  # TODO

**Written answer.**
In three to five sentences, report race day: how many races were clean, the median and the mean, and whether you met the bar.
Compare the race with your plan: how far is the median from `planned_race_frozen`, and is that gap what a three-lap race from a standing start would add?
If a race was not clean, say what happened in it (when the car crashed, and how it finished) from its `RaceResult`.

*Your answer here.*

## Question 6 - What bought what

After Question 5 your teacher hands out the **race-day record**: the reference agents on the same five sealed cars, with the race-day line and grip, at several fractions, including Unit 10's agent (no fallback) at 0.54.
Use it, and your own race day, to answer in three parts.
No new races.

**Written answer.**

1. **The margin.** What did the margin buy on race day?
   Compare the fraction the margin rule picks with the fractions above it, race by race: clean count, race times, mean.
2. **The fallback.** What did the fallback buy at 0.54 (Unit 10's agent against Unit 11's, the same cars)?
   Why did it still not make 0.54 a good choice for this bar?
3. **The expected time.** Why did the expected race time over the training family misjudge race day?
   Name what the family's record could not contain.
   Say which of your Question 3 assumptions race day tested.

*Your answer here.*